# Life-Link Smart Traffic System — Vehicle Demo Notebook

This notebook demonstrates:
1. Vehicle physics simulation with V2I packet generation
2. Broker communication
3. Sample JSON packets
4. CSV log export
5. Kinematics plots

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import json, time
import src.config as cfg
from src.comm.broker import Broker
from src.vehicle.vehicle import Vehicle
from src.vehicle.physics import braking_distance, calculate_eta, update_kinematics
from src.logging.logger import CSVLogger
print('Life-Link modules loaded successfully!')

## 1. Spawn N Vehicles and Show Sample Packets

In [ ]:
N_VEHICLES = 8
broker = Broker()
vehicles = []

import random
random.seed(42)
lanes = cfg.LANE_IDS
vtypes = ['car', 'bike', 'auto', 'ambulance']

for i in range(N_VEHICLES):
    lane  = lanes[i % 4]
    vtype = vtypes[i % 4] if i < 4 else random.choice(['car','bike','auto'])
    v = Vehicle(lane, vehicle_type=vtype, broker=broker, zone_id='Demo')
    vehicles.append(v)

print(f'Created {len(vehicles)} vehicles:')
for v in vehicles:
    print(f'  {v}')

In [ ]:
# Show sample packet from each unique vehicle type
print('=== SAMPLE V2I PACKETS ===')
seen_types = set()
for v in vehicles:
    if v.vehicle_type not in seen_types:
        seen_types.add(v.vehicle_type)
        pkt = v.to_packet()
        print(f'\n--- {v.vehicle_type.upper()} ---')
        print(json.dumps(pkt, indent=2))

## 2. Simulate Physics and Broker Communication

In [ ]:
from src.controller.controller import IntersectionController

ctrl = IntersectionController(broker, zone_id='Demo')
dt   = cfg.DT
t    = 0.0
SIM_DURATION = 10.0

wait_log = {v.vehicle_id: [] for v in vehicles}
time_log = []

while t < SIM_DURATION:
    for v in vehicles:
        v.broadcast()
    ctrl.step(dt)
    sig = ctrl.get_signal_state()
    ns, ew = sig['NS'], sig['EW']
    for v in vehicles:
        lane_sig = ns if v.lane_id in ('north','south') else ew
        v.update(dt, lane_sig)
        wait_log[v.vehicle_id].append(v.wait_time)
    time_log.append(t)
    t += dt

print(f'Simulation complete: {SIM_DURATION}s simulated')
print(f'Broker total packets received: {broker.total_published}')
print(f'\nFinal signal state:')
print(json.dumps(ctrl.get_signal_state(), indent=2))

## 3. Export CSV Log

In [ ]:
import os
os.makedirs('../logs', exist_ok=True)
logger = CSVLogger('../logs/notebook_demo.csv')

for v in vehicles:
    logger.log_vehicle(v.vehicle_id, 'Demo', v.lane_id, v.wait_time, 'final_state')

logger.log_event('simulation_complete', zone_id='Demo', wait_time=SIM_DURATION)
print('CSV log exported to ../logs/notebook_demo.csv')

import csv
with open('../logs/notebook_demo.csv') as f:
    rows = list(csv.DictReader(f))
print(f'Total log rows: {len(rows)}')
print('\nFirst 5 rows:')
for row in rows[:5]:
    print(dict(row))

## 4. Kinematics Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

# Plot wait times per vehicle
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for vid, waits in wait_log.items():
    vtype = next(v.vehicle_type for v in vehicles if v.vehicle_id == vid)
    col = {'car':'#3498DB','bike':'#F39C12','auto':'#E67E22','ambulance':'#E74C3C'}.get(vtype,'gray')
    axes[0].plot(time_log, waits, label=f'{vtype} ({vid[:8]})', color=col, alpha=0.7)

axes[0].set_xlabel('Sim Time (s)')
axes[0].set_ylabel('Cumulative Wait Time (s)')
axes[0].set_title('Vehicle Wait Times During Simulation')
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)

# Physics: velocity curves
t_ax  = np.linspace(0, 15, 300)
dt_   = t_ax[1] - t_ax[0]

def sim_velocity(v0, a, max_v, n):
    vs = [v0]
    v = v0
    for _ in range(n-1):
        v = min(v + a * dt_, max_v)
        vs.append(v)
    return vs

axes[1].plot(t_ax, sim_velocity(0, 2.0, 14, len(t_ax)), label='Car (a=2.0, vmax=14)', color='#3498DB')
axes[1].plot(t_ax, sim_velocity(0, 2.0, 18, len(t_ax)), label='Bike (a=2.0, vmax=18)', color='#F39C12')
axes[1].plot(t_ax, sim_velocity(0, 2.5, 20, len(t_ax)), label='Ambulance (a=2.5, vmax=20)', color='#E74C3C')

axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Speed (m/s)')
axes[1].set_title('V2I Vehicle Speed Profiles')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Life-Link — Vehicle Kinematics Demo', fontsize=13, fontweight='bold')
plt.tight_layout()

os.makedirs('../output', exist_ok=True)
plt.savefig('../output/notebook_kinematics.png', dpi=120)
plt.show()
print('Plot saved to ../output/notebook_kinematics.png')

## 5. Physics Verification — Braking Distance and ETA

In [ ]:
print('=== Physics Verification ===')
print()

test_cases = [
    ('Car at 10 m/s',       10.0, 0.0,  5.0),
    ('Car at 14 m/s',       14.0, 0.0,  5.0),
    ('Ambulance at 20 m/s', 20.0, 0.0,  5.0),
    ('Car braking hard',    12.0, 0.0,  8.0),
]

print(f"{'Scenario':<28} {'Speed':>6} {'Brake Dist':>11} {'ETA @300m':>10}")
print('-' * 60)
for name, v, a, decel in test_cases:
    bd  = braking_distance(v, decel)
    eta = calculate_eta(300.0, v, a)
    print(f'{name:<28} {v:>5.1f}m/s  {bd:>8.2f}m   {eta:>8.2f}s')

print()
print(f'Detection Zone radius: {cfg.DETECTION_RADIUS}m')
print(f'Yellow duration:       {cfg.YELLOW_DURATION}s (mandatory)')
print(f'Min green time:        {cfg.MIN_GREEN_TIME}s')
print(f'Max green time:        {cfg.MAX_GREEN_TIME}s')
print(f'Recovery green:        {cfg.RECOVERY_GREEN}s')